# Imports

In [149]:
import pandas as pd
import numpy as np

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

import os

import sqlite3
import faiss

# Load data

In [150]:
CSV_PATH = "../data/csv/processed/"
files    = os.listdir(CSV_PATH)
csvs     = sorted([file.split('_')[1].split('.')[0] for file in files])

historic_csvs = csvs[:-1]
train_df      = pd.concat([pd.read_csv(f"{CSV_PATH}products_{csv}.csv") for csv in historic_csvs], ignore_index=True)

latest_csv = csvs[-1]
test_df    = pd.read_csv(f"{CSV_PATH}products_{latest_csv}.csv")

In [151]:
df = pd.read_csv("../data/csv/mail_groceries.csv")

conn = sqlite3.connect("../data/sql/swipes.db")
labels = pd.read_sql_query("SELECT data_id, is_liked, is_superliked, is_passed FROM swipes", conn)
conn.close()

In [152]:
labels

,data_id,is_liked,is_superliked,is_passed
0,10862993,1,0,0
1,10876624,1,0,0
2,10834070,0,1,0
3,10848222,1,0,0
4,10820980,1,0,0
...,...,...,...,...
571,10834055,1,0,0
572,10863003,1,0,0
573,10848105,1,0,0
574,10834062,1,0,0


# Pre-processing 

In [153]:
model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")

In [154]:
passage_train = train_df['translated_product'].to_numpy()
passage_embeddings_train = model.encode(passage_train)

passage_test = test_df['translated_product'].to_numpy()
passage_embeddings_test = model.encode(passage_test)

In [155]:
onehot = ['category']
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown='ignore'), onehot)
])
X = preprocessor.fit_transform(train_df).toarray()
num_categories = len(preprocessor.named_transformers_['cat'].categories_[0])
print(f"Number of categories after one-hot encoding: {num_categories}")

Number of categories after one-hot encoding: 246


In [156]:
train_df_processed = pd.DataFrame(X, columns=preprocessor.get_feature_names_out())
train_df_processed = pd.merge(train_df, train_df_processed, left_index=True, right_index=True)
cat_features_train = train_df_processed.values[:,-num_categories:]
X_final_train = np.hstack((passage_embeddings_train, cat_features_train))

test_df_processed = pd.DataFrame(preprocessor.transform(test_df).toarray(), columns=preprocessor.get_feature_names_out())
test_df_processed = pd.merge(test_df, test_df_processed, left_index=True, right_index=True)
cat_features_test = test_df_processed.values[:,-num_categories:]
X_final_test = np.hstack((passage_embeddings_test, cat_features_test))

# Train/test split

In [166]:
# train, val = train_test_split(X_final_train, test_size=0.2, random_state=42)
threshold = int(0.8 * X_final_train.shape[0])
val = X_final_train[threshold:]
train = X_final_train[:threshold]
test = X_final_test

# Create DataFrames with embeddings + category features
embedding_cols = [f'emb_{i}' for i in range(passage_embeddings_train.shape[1])]
cat_cols = preprocessor.get_feature_names_out().tolist()
all_cols = embedding_cols + cat_cols

train_df_split = pd.DataFrame(train, columns=all_cols)
val_df_split = pd.DataFrame(val, columns=all_cols)
test_df_split = pd.DataFrame(test, columns=all_cols)

# Merge with original metadata (data_id, product_name, etc.)
train_indices = train_df_processed.index[~train_df_processed.index.isin(val_df_split.index)].tolist()
val_indices   = train_df_processed.index[train_df_processed.index.isin(val_df_split.index)].tolist()

train = pd.concat([train_df_processed.iloc[train_indices].reset_index(drop=True), train_df_split], axis=1)
val = pd.concat([train_df_processed.iloc[val_indices].reset_index(drop=True), val_df_split], axis=1)
test = pd.concat([test_df_processed.reset_index(drop=True), test_df_split], axis=1)

In [167]:
train.shape, val.shape, test.shape

((1646, 1531), (412, 1531), (303, 1531))

# FAISS

In [178]:
# Build FAISS index for cosine similarity over translated product embeddings
train_vecs = np.ascontiguousarray(train_df_split.astype('float32'))
test_vecs  = np.ascontiguousarray(test_df_split.astype('float32'))

# Normalize for cosine similarity (dot-product index)
faiss.normalize_L2(train_vecs)
faiss.normalize_L2(test_vecs)

index = faiss.IndexFlatIP(train_vecs.shape[1])
index.add(train_vecs)
print(f"Indexed {index.ntotal} product vectors")

# Example lookup: top-5 nearest neighbors for first test item
D, I = index.search(test_vecs[200].reshape(1,-1), 5)
print("Top-5 neighbors (indices):", I[0])
print("Similarity scores:", D[0])

Indexed 1646 product vectors
Top-5 neighbors (indices): [1552 1551 1550  715  714]
Similarity scores: [1. 1. 1. 1. 1.]


In [179]:
test_df.iloc[200].values

array([np.int64(10876658), np.float64(129.0), 'Espresso Forza', 'Kaffe',
       'Kaffebønner', np.int64(1), np.int64(1), 'pk.', 'Netto',
       'https://static.tilbudsugen.dk/1st-retail/2025/51/64202/Netto512025-1_24_2492x4727_4998x6802_zoom.jpg',
       '2025-12-13', '2025-12-19',
       'https://res.cloudinary.com/dfqzmnlga/image/upload/v1765903610/2025-12-13/10876658.jpg',
       'Coffee beans',
       '☕️ The world needs more bold, bitter, and unforgettable moments.'],
      dtype=object)

In [181]:
train_df.iloc[1550].values

array([np.int64(10848045), np.float64(139.0), 'Qualita Rossa', 'Kaffe',
       'Kaffebønner', np.int64(1), np.int64(1), 'ps.', 'Netto',
       '2025-11-29', '2025-12-05',
       'https://res.cloudinary.com/dfqzmnlga/image/upload/v1764415600/2025-11-29/10848045.jpg',
       'Coffee beans', '☕️ The world needs more coffee wisdom.',
       'https://static.tilbudsugen.dk/1st-retail/2025/49/64080/Netto492025-1_18_2443x3987_4886x6879_zoom.jpg'],
      dtype=object)